# Traffic Signal Control — Statistics Visualization Notebook

This notebook visualizes your experiment outputs using **line charts**:
- **Metrics vs episode** (from `output/eval_*.json`)
- **Metrics vs time** (from `output/vehicle_data*.csv`, `output/traffic_light_data*.csv`, `output/detector_data*.csv`)

> Notes
- Run this notebook from the project root: `D:\Final Year Project\traffic-signal-control`
- If you use conda, ensure you launch Jupyter from the same env you used for running SUMO + analysis (your `pytorch_env`).


In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# Plotting: Plotly is already listed in your README; fallback to matplotlib if needed.
try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception as e:
    px = None
    go = None
    print("Plotly import failed:", e)

ROOT = Path(".").resolve()
OUT = ROOT / "output"


def load_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def list_files(pattern: str, base: Path = OUT) -> List[Path]:
    return sorted(base.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)


def require_plotly():
    if px is None or go is None:
        raise RuntimeError("Plotly is not available. Install it in your env: pip install plotly")


print("ROOT:", ROOT)
print("Found eval JSONs:", [p.name for p in list_files("eval_*.json")])
print("Found vehicle CSVs:", [p.name for p in list_files("*vehicle*_data*.csv")])
print("Found traffic light CSVs:", [p.name for p in list_files("*traffic*_light*_data*.csv")])
print("Found detector CSVs:", [p.name for p in list_files("*detector*_data*.csv")])


ROOT: D:\Final Year Project\traffic-signal-control
Found eval JSONs: ['eval_vancouver.json', 'eval_los_angeles.json']
Found vehicle CSVs: ['vehicle_data.csv']
Found traffic light CSVs: ['traffic_light_data.csv']
Found detector CSVs: ['detector_data.csv']


## 1) Metrics vs Episode (Evaluation JSON)

Use this when you want to show **learning stability or performance per episode**.

Your repo already has examples:
- `output/eval_los_angeles.json`
- `output/eval_vancouver.json`


In [2]:
require_plotly()

EVAL_FILES = {
    p.stem.replace("eval_", ""): p
    for p in list_files("eval_*.json")
}
EVAL_FILES


{'vancouver': WindowsPath('D:/Final Year Project/traffic-signal-control/output/eval_vancouver.json'),
 'los_angeles': WindowsPath('D:/Final Year Project/traffic-signal-control/output/eval_los_angeles.json')}

In [3]:
def eval_results_to_df(eval_json: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for ep in (eval_json.get("results") or []):
        metrics = ep.get("metrics") or {}
        row = {
            "episode": ep.get("episode"),
            "reward_sum": ep.get("reward_sum"),
            "avg_reward": ep.get("avg_reward"),
            "wall_seconds": ep.get("wall_seconds"),
            "decisions": ep.get("decisions"),
            **metrics,
        }
        rows.append(row)
    df = pd.DataFrame(rows)
    # Normalize dtypes
    if "episode" in df.columns:
        df["episode"] = pd.to_numeric(df["episode"], errors="coerce")
    return df.sort_values("episode")


def plot_metric_vs_episode(df: pd.DataFrame, metric: str, title: str = ""):
    if metric not in df.columns:
        print(f"Metric '{metric}' not found. Available columns:\n", sorted(df.columns))
        return
    fig = px.line(
        df,
        x="episode",
        y=metric,
        markers=True,
        title=title or f"{metric} vs episode",
    )
    fig.update_layout(xaxis_title="Episode", yaxis_title=metric)
    fig.show()


def plot_many_metrics_vs_episode(df: pd.DataFrame, metrics: List[str], label: str):
    # One figure, multiple lines
    keep = [m for m in metrics if m in df.columns]
    missing = [m for m in metrics if m not in df.columns]
    if missing:
        print("Missing metrics:", missing)
    if not keep:
        raise ValueError("No metrics found to plot")

    long = df.melt(id_vars=["episode"], value_vars=keep, var_name="metric", value_name="value")
    fig = px.line(
        long,
        x="episode",
        y="value",
        color="metric",
        markers=True,
        title=f"{label}: metrics vs episode",
    )
    fig.update_layout(xaxis_title="Episode", yaxis_title="Metric value")
    fig.show()


# Pick which eval file to visualize
DATASET_KEY = "los_angeles"  # change to 'vancouver' if you want
path = EVAL_FILES.get(DATASET_KEY)
assert path is not None, f"No eval file for '{DATASET_KEY}'. Available: {list(EVAL_FILES.keys())}"

eval_json = load_json(path)
df_ep = eval_results_to_df(eval_json)
df_ep.head()


,episode,reward_sum,avg_reward,wall_seconds,decisions,avg_waiting_time,max_waiting_time,total_waiting_time,avg_max_waiting_time_per_vehicle,vehicles_with_waiting,...,avg_fuel_per_vehicle,total_nox,total_pmx,avg_lane_occupancy,lane_occupancy_std,max_lane_occupancy,avg_acceleration,acceleration_std,harsh_braking_events,vehicle_type_stats
0,1,-213.37,-0.444521,14.433734,480,17.342614,299.0,435421.0,36.285207,797,...,0.0,0.0,0.0,6.772313,11.210692,62.187106,0.054107,1.284046,1260,"{'('waiting_time', 'mean')': {'bus_bus': 21.92..."
1,2,-213.37,-0.444521,14.094065,480,17.342614,299.0,435421.0,36.285207,797,...,0.0,0.0,0.0,6.772313,11.210692,62.187106,0.054107,1.284046,1260,"{'('waiting_time', 'mean')': {'bus_bus': 21.92..."
2,3,-213.37,-0.444521,14.106775,480,17.342614,299.0,435421.0,36.285207,797,...,0.0,0.0,0.0,6.772313,11.210692,62.187106,0.054107,1.284046,1260,"{'('waiting_time', 'mean')': {'bus_bus': 21.92..."
3,4,-213.37,-0.444521,14.253954,480,17.342614,299.0,435421.0,36.285207,797,...,0.0,0.0,0.0,6.772313,11.210692,62.187106,0.054107,1.284046,1260,"{'('waiting_time', 'mean')': {'bus_bus': 21.92..."
4,5,-213.37,-0.444521,14.032633,480,17.342614,299.0,435421.0,36.285207,797,...,0.0,0.0,0.0,6.772313,11.210692,62.187106,0.054107,1.284046,1260,"{'('waiting_time', 'mean')': {'bus_bus': 21.92..."


In [4]:
# Common paper-style metrics (you can add/remove)
KEY_METRICS = [
    "avg_waiting_time",
    "avg_queue_length",
    "throughput_per_hour",
    "avg_speed",
    "congestion_index",
    "avg_pressure",
]

plot_many_metrics_vs_episode(df_ep, KEY_METRICS + ["reward_sum"], label=DATASET_KEY)


## 2) Metrics vs Time (Time-series CSV)

Use this when you want to show **system dynamics** during one simulation run (e.g., queue build-up and dissipation).

This notebook will compute time-series aggregates from `output/vehicle_data.csv`:
- mean waiting time over time
- mean speed over time
- an approximate queue count over time (vehicles with waiting time > 5s)
- optional emissions over time (if columns exist)


In [5]:
def load_vehicle_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    # normalize expected columns
    for col in ["time", "waiting_time", "speed"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def vehicle_timeseries(df: pd.DataFrame, queue_wait_threshold_s: float = 5.0) -> pd.DataFrame:
    if "time" not in df.columns:
        raise ValueError("vehicle CSV must include a 'time' column")

    out = pd.DataFrame({"time": sorted(df["time"].dropna().unique())})

    # mean waiting time over all active vehicles at time t
    if "waiting_time" in df.columns:
        out["avg_waiting_time_t"] = df.groupby("time")["waiting_time"].mean().reindex(out["time"]).values
        out["p95_waiting_time_t"] = df.groupby("time")["waiting_time"].quantile(0.95).reindex(out["time"]).values
    else:
        out["avg_waiting_time_t"] = np.nan
        out["p95_waiting_time_t"] = np.nan

    # mean speed
    if "speed" in df.columns:
        out["avg_speed_t"] = df.groupby("time")["speed"].mean().reindex(out["time"]).values
    else:
        out["avg_speed_t"] = np.nan

    # queue proxy: count vehicles with waiting_time > threshold
    if "waiting_time" in df.columns:
        queued = df[df["waiting_time"] > float(queue_wait_threshold_s)]
        out["queued_vehicle_count_t"] = queued.groupby("time").size().reindex(out["time"], fill_value=0).values
    else:
        out["queued_vehicle_count_t"] = np.nan

    # emissions (if present)
    emission_cols = [c for c in ["co2_emission", "fuel_consumption", "nox_emission", "pmx_emission"] if c in df.columns]
    for c in emission_cols:
        out[f"sum_{c}_t"] = df.groupby("time")[c].sum().reindex(out["time"], fill_value=0.0).values

    return out


def plot_timeseries(ts: pd.DataFrame, y: str, title: str = "", smooth_window: int = 15):
    if y not in ts.columns:
        print(f"Missing column: {y}")
        return

    plot_df = ts[["time", y]].copy()
    if smooth_window and smooth_window > 1:
        plot_df[f"{y}_smooth"] = plot_df[y].rolling(smooth_window, min_periods=1).mean()
        fig = px.line(plot_df, x="time", y=[y, f"{y}_smooth"], title=title or f"{y} vs time")
    else:
        fig = px.line(plot_df, x="time", y=y, title=title or f"{y} vs time")

    fig.update_layout(xaxis_title="Simulation time (s)", yaxis_title=y)
    fig.show()


veh_files = list_files("*vehicle*_data*.csv")
assert veh_files, "No vehicle_data CSV found under output/"
veh_path = veh_files[0]
print("Using vehicle CSV:", veh_path)

df_v = load_vehicle_csv(veh_path)
ts = vehicle_timeseries(df_v, queue_wait_threshold_s=5.0)
ts.head()


Using vehicle CSV: D:\Final Year Project\traffic-signal-control\output\vehicle_data.csv


,time,avg_waiting_time_t,p95_waiting_time_t,avg_speed_t,queued_vehicle_count_t,sum_co2_emission_t,sum_fuel_consumption_t,sum_nox_emission_t,sum_pmx_emission_t
0,5.0,0.400000,1.6,10.103090,0,0.0,0.0,0.0,0.0
1,10.0,1.250000,5.6,5.514559,1,0.0,0.0,0.0,0.0
2,15.0,1.727273,8.5,7.215333,1,0.0,0.0,0.0,0.0
3,20.0,1.785714,10.5,6.164400,2,0.0,0.0,0.0,0.0
4,25.0,2.529412,14.0,5.777929,3,0.0,0.0,0.0,0.0


In [ ]:
plot_timeseries(ts, "avg_waiting_time_t", title="Average waiting time vs time", smooth_window=15)
plot_timeseries(ts, "p95_waiting_time_t", title="P95 waiting time vs time", smooth_window=15)
plot_timeseries(ts, "queued_vehicle_count_t", title="Queued vehicles (>5s wait) vs time", smooth_window=15)
plot_timeseries(ts, "avg_speed_t", title="Average speed vs time", smooth_window=15)

# Optional emissions plots (only if these columns exist in the CSV)
for col in ["sum_co2_emission_t", "sum_fuel_consumption_t", "sum_nox_emission_t", "sum_pmx_emission_t"]:
    if col in ts.columns:
        plot_timeseries(ts, col, title=f"{col} vs time", smooth_window=15)


## 3) How to compare DQN/MARL against baselines (and which plots to use)

You usually compare against baselines in **two different situations**:

### A) Final performance comparison (best for the main Results table)
Use this when you have **multiple independent runs/episodes per strategy**.
- **What you compare**: `avg_waiting_time`, `avg_queue_length`, `throughput_per_hour`, `avg_speed` (and optionally emissions).
- **Best plot (recommended)**: box/violin plots (distribution) + mean/CI table.
- **If you must use line charts**: use a **line chart with markers + error bars** where x=**strategy** and y=**mean metric**.

### B) Dynamic behavior comparison (best for explaining why one method is better)
Use this when you want to show *how* congestion changes over time.
- Plot **queue vs time**, **waiting time vs time**, **speed vs time** for DQN vs baselines.
- This requires you to have per-strategy CSV logs (one run per strategy) or multiple runs you can average.

### C) Learning / convergence (only for RL methods)
Use this to prove your RL training/evaluation is stable.
- Plot **reward vs episode** and optionally **loss vs training step**.

> Rule of thumb
- If you want to claim “DQN is better”: show **A (final metrics)**.
- If you want to explain “why it’s better”: add **B (time-series)**.
- If reviewers question RL stability: add **C (episode curves)**.


In [ ]:
# --- Strategy comparison helper (line+markers with error bars) ---
#
# This expects you to provide multiple runs per strategy.
# You can source those runs from:
# - multiple eval JSONs (one per strategy), OR
# - an academic batch JSON, OR
# - your own list of metrics dicts.


def summarize_runs(runs: List[Dict[str, Any]], metric: str) -> Dict[str, float]:
    vals = []
    for r in runs:
        if metric in r and r[metric] is not None:
            try:
                vals.append(float(r[metric]))
            except Exception:
                pass
    if not vals:
        return {"mean": float("nan"), "std": float("nan"), "n": 0}
    arr = np.array(vals, dtype=float)
    return {"mean": float(arr.mean()), "std": float(arr.std(ddof=1)) if len(arr) > 1 else 0.0, "n": int(len(arr))}


def plot_strategy_comparison_line(strategy_to_runs: Dict[str, List[Dict[str, Any]]], metric: str, title: str = ""):
    require_plotly()

    rows = []
    for strat, runs in strategy_to_runs.items():
        s = summarize_runs(runs, metric)
        rows.append({"strategy": strat, **s})

    df = pd.DataFrame(rows)
    df = df.sort_values("strategy")

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df["strategy"],
            y=df["mean"],
            mode="lines+markers",
            name=metric,
            error_y=dict(type="data", array=df["std"], visible=True),
        )
    )

    fig.update_layout(
        title=title or f"{metric}: mean ± std across strategies",
        xaxis_title="Strategy",
        yaxis_title=f"{metric} (mean ± std)",
    )
    fig.show()


# Example template: plug in your own results.
#
# For baselines, you typically create results by running each strategy multiple times
# (or using the Flask 'Academic Experiments' page), then saving per-run metrics.
#
# Here is a minimal placeholder structure:
strategy_to_runs_example = {
    # "FIXED_TIME": [ {"avg_waiting_time": ...}, {"avg_waiting_time": ...}, ... ],
    # "MAX_PRESSURE": [ ... ],
    # "DQN": [ ... ],
}

strategy_to_runs_example


In [ ]:
# OPTIONAL: auto-build strategy_to_runs from eval_*.json (if you have one eval JSON per strategy)
#
# Right now you only have eval_los_angeles.json and eval_vancouver.json (these are dataset evals).
# If later you create files like:
#   output/eval_los_angeles_FIXED_TIME.json
#   output/eval_los_angeles_MAX_PRESSURE.json
#   output/eval_los_angeles_DQN.json
# you can parse them here.

import re


def parse_eval_strategy_filename(p: Path) -> Tuple[Optional[str], Optional[str]]:
    """Return (dataset, strategy) if filename matches eval_<dataset>_<strategy>.json"""
    m = re.match(r"^eval_(?P<dataset>.+?)_(?P<strategy>[A-Z0-9_]+)\.json$", p.name)
    if not m:
        return None, None
    return m.group("dataset"), m.group("strategy")


all_eval = list_files("eval_*.json")
auto = {}
for p in all_eval:
    dataset, strategy = parse_eval_strategy_filename(p)
    if dataset is None:
        continue
    data = load_json(p)
    df = eval_results_to_df(data)
    # each row already contains metrics, so convert to list-of-dicts
    auto.setdefault(dataset, {})[strategy] = df.to_dict(orient="records")

auto  # if empty, that's normal until you create per-strategy eval files


## 4) Training curves from saved models (.pt)

Use this when you want to show **learning convergence** (reward/loss/epsilon over training episodes).

This reads `models/*.pt` checkpoints and plots:
- episode reward vs episode
- epsilon vs episode
- loss vs training step (smoothed)


In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_FILES = sorted(MODEL_DIR.glob("*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
print("Model checkpoints:", [p.name for p in MODEL_FILES])

try:
    import torch
except Exception as e:
    torch = None
    print("torch import failed (training-curve section will be skipped):", e)


In [ ]:
def load_training_history_from_checkpoint(path: Path) -> Optional[Dict[str, Any]]:
    if torch is None:
        return None
    ckpt = torch.load(path, map_location="cpu")
    hist = ckpt.get("training_history") or {}
    return {
        "path": str(path),
        "episode_count": ckpt.get("episode_count"),
        "step_count": ckpt.get("step_count"),
        "epsilon": ckpt.get("epsilon"),
        "training_history": hist,
    }


def training_history_to_dfs(hist: Dict[str, Any]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return (episode_df, loss_df)."""
    th = hist.get("training_history") or {}

    rewards = th.get("episode_rewards") or []
    epsilons = th.get("epsilon_values") or []
    lengths = th.get("episode_lengths") or []

    # Episode-level df
    ep_n = max(len(rewards), len(epsilons), len(lengths))
    ep_df = pd.DataFrame({
        "episode": np.arange(1, ep_n + 1, dtype=int),
        "episode_reward": pd.Series(rewards, dtype=float) if rewards else np.nan,
        "epsilon": pd.Series(epsilons, dtype=float) if epsilons else np.nan,
        "episode_length": pd.Series(lengths, dtype=float) if lengths else np.nan,
    })

    # Step-level losses (can be very long)
    losses = th.get("losses") or []
    loss_df = pd.DataFrame({
        "train_step": np.arange(1, len(losses) + 1, dtype=int),
        "loss": pd.Series(losses, dtype=float) if losses else np.nan,
    })

    return ep_df, loss_df


def plot_training_curves(ckpt_path: Path, loss_smooth: int = 500):
    require_plotly()
    if torch is None:
        raise RuntimeError("torch not available in this kernel")

    info = load_training_history_from_checkpoint(ckpt_path)
    if not info:
        return

    ep_df, loss_df = training_history_to_dfs(info)
    title_prefix = ckpt_path.name

    # Reward vs episode
    if "episode_reward" in ep_df.columns and ep_df["episode_reward"].notna().any():
        tmp = ep_df[["episode", "episode_reward"]].copy()
        tmp["reward_smooth"] = tmp["episode_reward"].rolling(10, min_periods=1).mean()
        fig = px.line(tmp, x="episode", y=["episode_reward", "reward_smooth"], title=f"{title_prefix}: reward vs episode")
        fig.update_layout(xaxis_title="Episode", yaxis_title="Episode reward")
        fig.show()

    # Epsilon vs episode
    if "epsilon" in ep_df.columns and ep_df["epsilon"].notna().any():
        fig = px.line(ep_df, x="episode", y="epsilon", title=f"{title_prefix}: epsilon vs episode")
        fig.update_layout(xaxis_title="Episode", yaxis_title="Epsilon")
        fig.show()

    # Loss vs training step (downsample + smooth)
    if "loss" in loss_df.columns and loss_df["loss"].notna().any() and len(loss_df) > 0:
        # Downsample for speed
        step = max(1, len(loss_df) // 5000)
        d = loss_df.iloc[::step].copy()
        d["loss_smooth"] = d["loss"].rolling(loss_smooth // step if loss_smooth else 1, min_periods=1).mean()
        fig = px.line(d, x="train_step", y=["loss", "loss_smooth"], title=f"{title_prefix}: loss vs training step")
        fig.update_layout(xaxis_title="Training step", yaxis_title="Loss")
        fig.show()


# Pick a checkpoint to visualize
if MODEL_FILES:
    CKPT = MODEL_FILES[0]
    print("Using checkpoint:", CKPT)
    if torch is not None:
        plot_training_curves(CKPT)


## 5) Multi-strategy overlays (line charts)

This section helps you overlay **DQN vs baselines** on the *same* line chart.

### 5.1 Metrics vs episode (treat episodes as independent runs)
This is useful when you run each strategy multiple times and store one metric dict per run.

### 5.2 Metrics vs time (overlay multiple CSV logs)
This is useful when you have one `vehicle_data_*.csv` per strategy.

> Practical file naming suggestion
- Save your outputs as:
  - `output/vehicle_data_<DATASET>_<STRATEGY>_run1.csv`
  - `output/vehicle_data_<DATASET>_<STRATEGY>_run2.csv`
  - etc.
Then this notebook can auto-group them.


In [ ]:
import re


def group_vehicle_csvs_by_strategy(files: List[Path]) -> Dict[str, List[Path]]:
    """Group vehicle CSVs by a <STRATEGY> token if filename contains it.

    Expected patterns (examples):
      - vehicle_data_los_angeles_DQN_run1.csv
      - vehicle_data_custom_FIXED_TIME_20260124.csv

    If no strategy token can be found, file goes under 'UNKNOWN'.
    """
    known = [
        "FIXED_TIME",
        "ADAPTIVE",
        "MAX_PRESSURE",
        "DQN",
        "MARL_DQN",
        "PRESSLIGHT",
        "GA",
        "GA_INTERNAL",
    ]

    out: Dict[str, List[Path]] = {}
    for p in files:
        name = p.name.upper()
        picked = None
        for k in known:
            if f"_{k}_" in name or name.endswith(f"_{k}.CSV") or f"{k}" in name:
                picked = k
                break
        out.setdefault(picked or "UNKNOWN", []).append(p)

    # sort each group by mtime (newest first)
    for k in out:
        out[k] = sorted(out[k], key=lambda x: x.stat().st_mtime, reverse=True)
    return out


def plot_timeseries_overlay(ts_by_label: Dict[str, pd.DataFrame], y: str, smooth_window: int = 15, title: str = ""):
    require_plotly()
    rows = []
    for label, ts_df in ts_by_label.items():
        if y not in ts_df.columns:
            continue
        d = ts_df[["time", y]].copy()
        if smooth_window and smooth_window > 1:
            d[y] = d[y].rolling(smooth_window, min_periods=1).mean()
        d["label"] = label
        rows.append(d)

    if not rows:
        print(f"No data found for '{y}'. Available columns per label:")
        for label, ts_df in ts_by_label.items():
            print("-", label, ":", sorted(ts_df.columns))
        return

    long = pd.concat(rows, ignore_index=True)
    fig = px.line(long, x="time", y=y, color="label", title=title or f"{y} vs time (overlay)")
    fig.update_layout(xaxis_title="Simulation time (s)", yaxis_title=y)
    fig.show()


# Auto-detect multiple vehicle CSVs and group them by strategy token
veh_files_all = list_files("*vehicle*_data*.csv")
groups = group_vehicle_csvs_by_strategy(veh_files_all)
print({k: [p.name for p in v[:3]] for k, v in groups.items()})

# Build one time-series per strategy using the newest file in that group
# (You can change to v[0], v[1] to select different runs.)
ts_by_strategy = {}
for strat, files in groups.items():
    if not files:
        continue
    df = load_vehicle_csv(files[0])
    ts_by_strategy[strat] = vehicle_timeseries(df, queue_wait_threshold_s=5.0)

list(ts_by_strategy.keys())


In [ ]:
# Overlay examples (DQN vs baselines) — works once you have per-strategy vehicle_data_*.csv

plot_timeseries_overlay(ts_by_strategy, "queued_vehicle_count_t", smooth_window=15, title="Queued vehicles vs time (overlay)")
plot_timeseries_overlay(ts_by_strategy, "avg_waiting_time_t", smooth_window=15, title="Avg waiting time vs time (overlay)")
plot_timeseries_overlay(ts_by_strategy, "avg_speed_t", smooth_window=15, title="Avg speed vs time (overlay)")
